# Предобработка

### Загрузка библиотек

In [1]:
import numpy as np
import pandas as pd 
import psycopg2
import phik 
from datetime import datetime 
import math
from pathlib import Path
from config.settings import DB_CONFIG

# Координаты Кремля
CENTER_LAT, CENTER_LON = 55.7520, 37.6175


### Загрузка датасета

In [2]:
conn = psycopg2.connect(**DB_CONFIG)
query = '''
WITH unioned AS (
    SELECT cian_id, price, price_per_m2, mortgage_allowed, deal_conditions,
        region, municipality, district, lat, lon, metro_stations,
        rooms, is_studio, flat_type, total_area, living_area, kitchen_area, floor, total_floors,
        ceiling_height, renovation, bathrooms, balcony, window_view,
        is_apartments, year_built, building_type, parking, passenger_lifts, cargo_lifts, is_new_building,
        developer, residential_complex, completion_date, publication_date,
        seller_type, phone_protected, photos_count, views_total, views_today,
        first_seen_at, last_seen_at, TRUE AS is_live, NULL::date AS snapshot_date
    FROM listings
    UNION ALL
    SELECT cian_id, price, price_per_m2, mortgage_allowed, deal_conditions,
        region, municipality, district, lat, lon, metro_stations,
        rooms, is_studio, flat_type, total_area, living_area, kitchen_area, floor, total_floors,
        ceiling_height, renovation, bathrooms, balcony, window_view,
        is_apartments, year_built, building_type, parking, passenger_lifts, cargo_lifts, is_new_building,
        developer, residential_complex, completion_date, publication_date,
        seller_type, phone_protected, photos_count, views_total, views_today,
        first_seen_at, last_seen_at, FALSE AS is_live, snapshot_date
    FROM listings_archive
),
lifecycle AS (
    SELECT cian_id,
        min(first_seen_at) AS first_seen,
        max(last_seen_at)  AS last_seen,
        bool_or(is_live)   AS still_active
    FROM unioned GROUP BY cian_id
),
latest AS (
    SELECT DISTINCT ON (cian_id)
        cian_id, price, mortgage_allowed, deal_conditions, region, municipality, district, lat, lon,
        COALESCE(jsonb_array_length(metro_stations), 0) AS n_metro,
        metro_stations -> 0 ->> 0 AS nearest_metro,
        (metro_stations -> 0 ->> 1)::int AS nearest_metro_time,
        (metro_stations -> 0 ->> 2 = 'walk') AS nearest_metro_walk,
        rooms, is_studio, flat_type, total_area, living_area, kitchen_area, floor, total_floors,
        ceiling_height, renovation, bathrooms, balcony, window_view,
        is_apartments, year_built, building_type, parking, passenger_lifts, cargo_lifts, is_new_building,
        developer, residential_complex, completion_date, publication_date,
        seller_type, phone_protected, photos_count, views_total, views_today
    FROM unioned
    ORDER BY cian_id, is_live DESC, snapshot_date DESC NULLS LAST
),
prices AS (
    SELECT cian_id,
        (array_agg(price ORDER BY recorded_at, id))[1] AS price_first,
        (array_agg(price ORDER BY recorded_at DESC, id DESC))[1] AS price_last,
        min(price) AS price_min, max(price) AS price_max, count(*) AS price_points
    FROM price_history GROUP BY cian_id
)
SELECT
    lc.first_seen, lc.last_seen,
    (lc.last_seen::date - lt.publication_date::date) AS days_on_market,
    (NOT lc.still_active)::int AS event_closed,
    lt.price, lt.mortgage_allowed, lt.deal_conditions, lt.region, lt.municipality, lt.district,
    lt.lat, lt.lon, lt.n_metro, lt.nearest_metro, lt.nearest_metro_time, lt.nearest_metro_walk,
    lt.rooms, lt.is_studio, lt.flat_type, lt.total_area, lt.living_area, lt.kitchen_area, lt.floor, lt.total_floors,
    lt.ceiling_height, lt.renovation, lt.bathrooms, lt.balcony, lt.window_view,
    lt.is_apartments, lt.year_built, lt.building_type, lt.parking, lt.passenger_lifts, lt.cargo_lifts, lt.is_new_building,
    lt.developer, lt.residential_complex, lt.completion_date, lt.publication_date,
    lt.seller_type, lt.phone_protected, lt.photos_count, lt.views_total, lt.views_today,
    COALESCE(pr.price_first, lt.price) AS price_first,
    COALESCE(pr.price_last, lt.price)  AS price_last,
    pr.price_min, pr.price_max, COALESCE(pr.price_points, 0) AS price_points
FROM lifecycle lc
JOIN latest lt USING (cian_id)
LEFT JOIN prices pr USING (cian_id)
ORDER BY lc.cian_id;
'''

df = pd.read_sql_query(query, conn)
conn.close()
print('До удаления "мусорных:"', df.shape)

df = df[df['event_closed'] == 1]
print('После удаления "мусорных:"', df.shape)


C:\Users\Nikita\AppData\Local\Temp\ipykernel_12524\2055985408.py:73: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


До удаления "мусорных:" (271387, 50)
После удаления "мусорных:" (162107, 50)


In [3]:
# маппинг бинарников
# из psycopg2 булевы приходят True/False, из csv-файла строками 't'/'f', поэтому мапим сразу оба варианта
bool_map = {'t': True, 'f': False, True: True, False: False}
for col in ['is_apartments', 'is_new_building', 'phone_protected', 'is_studio', 'mortgage_allowed', 'nearest_metro_walk']:
    df[col] = df[col].map(bool_map).astype('boolean')

df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
df['first_seen'] = pd.to_datetime(df['first_seen'], errors='coerce')

# фильтр на невалидные значения
filter_days = df['days_on_market'] >= 0
filter_area = df['total_area'] > 0 
filter_price = df['price'] > 0
df = df[filter_days & filter_area & filter_price]


# формирование признака расстояния до центра Москвыч
R = 6371.0 # радиус Земли в километрах
lat = np.radians(df['lat'])
lon = np.radians(df['lon'])
dlat = lat - math.radians(CENTER_LAT)
dlon = lon - math.radians(CENTER_LON)

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat) * math.cos(math.radians(CENTER_LAT)) 
    * np.sin(dlon / 2) ** 2
)
# в километрах
df['dist_to_center'] = 2 * R * np.arcsin(np.sqrt(a))


In [4]:
# у объявлений без метро ближайшая станция приходит пустой
df['nearest_metro'] = df['nearest_metro'].fillna('unknown')


In [5]:
# создание доп признаков
df['price_per_m2'] = df['price_first'] / df['total_area']

# обрезаем по 1му и 99му перцентилям
QUANT_TRESHOLD = 0.01
clip_cols = ['price', 'price_last', 'price_first', 'price_per_m2', 'total_area', 'living_area', 'kitchen_area']

for col in clip_cols:
    floor = df[col].quantile(QUANT_TRESHOLD)
    ceil = df[col].quantile(1 - QUANT_TRESHOLD)
    df = df[df[col].between(floor, ceil) | df[col].isna()]

df.to_csv('./data/listings_preprocessed.csv', index=False)